# Physiological Mismatch and Residual Consistency Network (PMRC-Net)

## Google Colab experiment notebook

This self-contained notebook implements **PMRC-Net v1.1** consistently for both experiment modes. It includes the v1.1 ordered asymmetric coupling, raw-scale physiological mismatch representation, and binary cross-entropy objective. It does not import project modules or use local files.

### Run guide

1. Open this notebook in Google Colab and select a GPU runtime if one is available.
2. Run all cells. At the upload prompt, choose the merged heart-disease CSV (for example, heart_disease_merged.csv).
3. Keep EXPERIMENT_MODE set to quick for a stratified train/validation/test experiment, or explicitly change it to nested for the five-seed nested evaluation.
4. Download the generated ZIP at the end. Run download_checkpoint() separately if you also want one checkpoint as a standalone file.

The established merged data contain **602 exact-row-deduplicated observations** (from 1,888 rows in the source CSV). These rows are not verified independent patients; row indices used for prediction alignment are not patient identifiers. Uploaded data are audited rather than assumed to match.

Quick mode reserves a stratified test split and uses only the training and validation partitions for fitting and early stopping. Nested mode follows the v1.1 design: five seeds, five outer folds per seed, four inner folds for training-duration selection, and a fresh outer-training refit. The outer test fold is not used for preprocessing, early stopping, checkpoint selection, or tuning.

All reported metrics and figures are computed from real held-out predictions and corresponding labels. The software sanity checks use synthetic tensors only and are explicitly not experiment results. Nothing in this notebook establishes patient independence, external validity, causal physiology, clinical readiness, or superiority.


## 2. Environment setup

Colab supplies the core packages used below. The notebook intentionally does not upgrade PyTorch, CUDA, or other packages. All training uses FP32.

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import math
import random
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

print(f"Python notebook initialized; PyTorch {torch.__version__}; NumPy {np.__version__}; pandas {pd.__version__}")


## 3. GPU/CPU detection

The selected Colab runtime determines hardware availability. CPU is supported. CUDA memory reporting is informational; training remains FP32.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    properties = torch.cuda.get_device_properties(DEVICE)
    total_gib = properties.total_memory / (1024 ** 3)
    free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
    print(f"Device: {DEVICE} ({properties.name})")
    print(f"GPU memory: {total_gib:.2f} GiB total; {free_bytes / (1024 ** 3):.2f} GiB currently free")
else:
    print(f"Device: {DEVICE}; CUDA is unavailable in this runtime.")


## 4. Dataset upload

Upload one CSV file directly from the Colab prompt. The uploaded file is read from memory; no Windows path, Drive mount, or repository file is required.

In [ ]:
from google.colab import files as colab_files

uploaded_files = colab_files.upload()
csv_uploads = {
    name: content for name, content in uploaded_files.items()
    if name.lower().endswith(".csv")
}
if len(csv_uploads) != 1:
    raise ValueError(f"Upload exactly one CSV file; received {len(csv_uploads)} CSV files.")

UPLOADED_FILENAME, UPLOADED_BYTES = next(iter(csv_uploads.items()))
try:
    RAW_DATAFRAME = pd.read_csv(io.BytesIO(UPLOADED_BYTES))
except (pd.errors.ParserError, UnicodeDecodeError, ValueError) as exc:
    raise ValueError(f"Could not parse uploaded file {UPLOADED_FILENAME!r} as a CSV.") from exc
if RAW_DATAFRAME.empty:
    raise ValueError("The uploaded CSV contains no data rows.")
print(f"Uploaded and parsed {UPLOADED_FILENAME!r}.")


## 5. Dataset inspection

Inspect the uploaded table before any transformations. Exact duplicate counts are computed on the raw rows, before categorical harmonization.

In [ ]:
print(f"Raw shape: {RAW_DATAFRAME.shape}")
print("Columns:", RAW_DATAFRAME.columns.tolist())
print(f"Raw exact duplicate rows: {int(RAW_DATAFRAME.duplicated().sum())}")
print("Missing values by column:")
display(RAW_DATAFRAME.isna().sum().rename("missing_count").to_frame())
print("First five records:")
display(RAW_DATAFRAME.head())


## 6. Data cleaning and preprocessing

The required columns, binary target, finite continuous values, and category vocabularies follow the v1.1 data configuration. Known source encodings are mapped explicitly: thal 0→0, 1→6, 2→3, 3→7, 6→6, 7→7; slope and ca use identity mappings over their documented codes. Unknown, missing, fractional, or non-finite categorical values are rejected. Continuous NaN/Inf values are rejected; no imputation is performed.

Exact duplicate rows are removed before encoding. The row_index assigned afterward is only a prediction-alignment index, not a patient ID. Each experiment fold fits StandardScaler on its training partition alone and applies that scaler to validation/test data.

In [ ]:
CONFIG = {
    "version": "1.1",
    "seed": 42,
    "seeds": [21, 42, 123, 2026, 3407],
    "data": {
        "target": "target",
        "deduplicate_exact_rows": True,
        "continuous_cols": ["age", "trestbps", "chol", "thalachh", "oldpeak"],
        "categorical_cols": ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"],
        "physiology_cols": ["age", "thalachh", "oldpeak", "exang"],
        "category_values": {
            "sex": [0, 1], "cp": [0, 1, 2, 3], "fbs": [0, 1],
            "restecg": [0, 1, 2], "exang": [0, 1], "slope": [0, 1, 2, 3],
            "ca": [0, 1, 2, 3, 4], "thal": [0, 3, 6, 7],
        },
        "pathway_map": {
            "metabolic": ["age", "sex", "trestbps", "chol", "fbs"],
            "exercise": ["thalachh", "oldpeak", "exang", "slope"],
            "coronary": ["cp", "restecg", "ca", "thal"],
        },
    },
    "model": {
        "hidden_dim": 16, "dropout": 0.1, "coupling": "asymmetric",
        "embedding_cardinality": 10,
    },
    "training": {
        "batch_size": 8, "learning_rate": 0.001, "weight_decay": 0.0001,
        "max_epochs": 100, "patience": 12, "min_delta": 0.0001,
        "gradient_clip_norm": 1.0, "inner_folds": 4, "outer_folds": 5,
        "num_workers": 0, "precision": "fp32",
    },
    "statistics": {"threshold": 0.5, "ece_bins": 10},
}

SOURCE_MAPPINGS = {
    "thal": {0: 0, 1: 6, 2: 3, 3: 7, 6: 6, 7: 7},
    "slope": {0: 0, 1: 1, 2: 2, 3: 3},
    "ca": {0: 0, 1: 1, 2: 2, 3: 3, 4: 4},
}

def validate_and_clean_dataset(raw_frame: pd.DataFrame, config: dict) -> tuple[pd.DataFrame, dict]:
    data_config = config["data"]
    target = data_config["target"]
    continuous = data_config["continuous_cols"]
    categorical = data_config["categorical_cols"]
    required = continuous + categorical + [target]
    missing_columns = sorted(set(required) - set(raw_frame.columns))
    if missing_columns:
        raise ValueError(f"CSV is missing required columns: {missing_columns}")

    raw_duplicate_count = int(raw_frame.duplicated().sum())
    frame = raw_frame.drop_duplicates().reset_index(drop=True).copy()

    for column in continuous:
        try:
            frame[column] = pd.to_numeric(frame[column], errors="raise")
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Continuous column {column!r} must contain numeric values.") from exc
        values = frame[column].to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            bad_row = int(np.argwhere(~np.isfinite(values))[0, 0])
            raise ValueError(f"Continuous column {column!r} contains a missing or non-finite value at deduplicated row {bad_row}.")

    for column in categorical:
        try:
            values = pd.to_numeric(frame[column], errors="raise")
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Categorical column {column!r} must contain numeric category codes.") from exc
        numeric_values = values.to_numpy(dtype=np.float64)
        if not np.isfinite(numeric_values).all() or not np.equal(numeric_values, np.floor(numeric_values)).all():
            raise ValueError(f"Categorical column {column!r} contains a missing, non-finite, or non-integer category code.")
        frame[column] = values.astype(np.int64)
        if column in SOURCE_MAPPINGS:
            mapping = SOURCE_MAPPINGS[column]
            invalid = ~frame[column].isin(mapping)
            if invalid.any():
                values_found = sorted(frame.loc[invalid, column].unique().tolist())
                raise ValueError(f"Invalid source category for {column!r}: {values_found[:10]}")
            frame[column] = frame[column].map(mapping).astype(np.int64)
        allowed = data_config["category_values"][column]
        invalid = ~frame[column].isin(allowed)
        if invalid.any():
            values_found = sorted(frame.loc[invalid, column].unique().tolist())
            raise ValueError(f"Invalid mapped category for {column!r}: {values_found[:10]}; expected one of {allowed}")

    try:
        frame[target] = pd.to_numeric(frame[target], errors="raise")
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Target column {target!r} must contain numeric binary labels.") from exc
    target_values = frame[target].to_numpy(dtype=np.float64)
    if not np.isfinite(target_values).all() or not np.isin(target_values, [0, 1]).all():
        raise ValueError(f"Target column {target!r} must contain only non-missing binary labels 0 and 1.")
    frame[target] = frame[target].astype(np.int64)
    if frame[target].nunique() != 2:
        raise ValueError("Both target classes 0 and 1 must be present.")

    frame.insert(0, "row_index", np.arange(len(frame), dtype=np.int64))
    audit = {
        "raw_rows": int(len(raw_frame)),
        "raw_exact_duplicate_rows": raw_duplicate_count,
        "deduplicated_rows": int(len(frame)),
        "missing_values_in_raw": int(raw_frame.isna().sum().sum()),
        "target_counts": {str(int(key)): int(value) for key, value in frame[target].value_counts().sort_index().items()},
        "category_values_after_mapping": {
            column: sorted(int(value) for value in frame[column].unique())
            for column in categorical
        },
    }
    return frame, audit

DATAFRAME, DATA_AUDIT = validate_and_clean_dataset(RAW_DATAFRAME, CONFIG)
DATA_SHA256 = hashlib.sha256(UPLOADED_BYTES).hexdigest()
print(json.dumps(DATA_AUDIT, indent=2))
print("Continuous features:", CONFIG["data"]["continuous_cols"])
print("Categorical features:", CONFIG["data"]["categorical_cols"])
print("Pathway map:", json.dumps(CONFIG["data"]["pathway_map"], indent=2))
if len(DATAFRAME) == 602:
    print("Validated reference size: 602 exact-row-deduplicated observations.")
else:
    print(f"Notice: uploaded data have {len(DATAFRAME)} exact-row-deduplicated rows; the established reference dataset has 602.")


## 7. PMRC-Net v1.1 architecture

The model implementation below is embedded from the authoritative v1.1 source. It contains three pathway encoders; a raw-scale physiological mismatch module; symmetric or ordered asymmetric pairwise coupling; mechanistic, interaction, and residual logits; and their additive final logit. The probability is sigmoid(z_total). The v1.1 objective is BCE-with-logits, with AdamW weight decay; the v1.0 empirical-direction hinge is not combined with it.

In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class ClinicalPhysiologyModule(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.a1 = nn.Parameter(torch.full((1,), 0.5))
        self.a2 = nn.Parameter(torch.full((1,), 0.5))
        self.a3 = nn.Parameter(torch.full((1,), 0.5))

    def forward(
        self,
        age_raw: torch.Tensor,
        max_hr_raw: torch.Tensor,
        oldpeak_raw: torch.Tensor,
        exang_raw: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        predicted_hr = 208.0 - 0.7 * age_raw
        utilization = max_hr_raw / torch.clamp(predicted_hr, min=1.0)
        heart_rate_deficit = torch.clamp(1.0 - utilization, min=0.0)
        ischemia = F.softplus(
            torch.clamp(self.a1, min=0.0) * oldpeak_raw
            + torch.clamp(self.a2, min=0.0) * exang_raw
            + torch.clamp(self.a3, min=0.0) * oldpeak_raw * exang_raw
        )
        return ischemia * heart_rate_deficit, heart_rate_deficit, ischemia


class PMRCNet(nn.Module):
    """PMRC-Net v1.1 with ordered asymmetric pairwise interactions."""

    def __init__(self, config: dict, model_config: dict | None = None) -> None:
        super().__init__()
        self.config = config
        self.model_config = model_config or config["model"]
        self.hidden_dim = int(self.model_config.get("hidden_dim", 16))
        self.dropout_rate = float(self.model_config.get("dropout", 0.1))
        self.coupling_mode = str(self.model_config.get("coupling", "asymmetric"))
        if self.coupling_mode not in {"asymmetric", "symmetric", "none"}:
            raise ValueError(f"Unsupported coupling mode: {self.coupling_mode}")
        data_config = config["data"]
        continuous = data_config["continuous_cols"]
        categorical = data_config["categorical_cols"]
        self.continuous_cols = list(continuous)
        self.categorical_cols = list(categorical)
        self.category_values = {
            name: tuple(int(value) for value in values)
            for name, values in data_config["category_values"].items()
        }
        if set(self.category_values) != set(self.categorical_cols):
            raise ValueError("category_values must define every categorical column exactly once")
        pathways = data_config["pathway_map"]
        self.metabolic_cols, self.exercise_cols, self.coronary_cols = (
            pathways[name] for name in ("metabolic", "exercise", "coronary")
        )
        self.metabolic_cont_indices = [continuous.index(c) for c in self.metabolic_cols if c in continuous]
        self.metabolic_cat_indices = [categorical.index(c) for c in self.metabolic_cols if c in categorical]
        self.exercise_cont_indices = [continuous.index(c) for c in self.exercise_cols if c in continuous]
        self.exercise_cat_indices = [categorical.index(c) for c in self.exercise_cols if c in categorical]
        self.coronary_cont_indices = [continuous.index(c) for c in self.coronary_cols if c in continuous]
        self.coronary_cat_indices = [categorical.index(c) for c in self.coronary_cols if c in categorical]
        self.embedding_cardinality = int(self.model_config.get("embedding_cardinality", 10))
        self.embeddings = nn.ModuleDict({
            column: nn.Embedding(self.embedding_cardinality, self.hidden_dim)
            for column in categorical
        })
        self.proj_metabolic = nn.Linear(
            len(self.metabolic_cont_indices) + len(self.metabolic_cat_indices) * self.hidden_dim,
            self.hidden_dim,
        )
        self.proj_exercise = nn.Linear(
            len(self.exercise_cont_indices) + len(self.exercise_cat_indices) * self.hidden_dim,
            self.hidden_dim,
        )
        self.proj_coronary = nn.Linear(
            len(self.coronary_cont_indices) + len(self.coronary_cat_indices) * self.hidden_dim,
            self.hidden_dim,
        )
        self.dropout = nn.Dropout(self.dropout_rate)
        self.phys_module = ClinicalPhysiologyModule()
        self.proj_me = nn.Linear(1, self.hidden_dim)
        if self.coupling_mode in {"asymmetric", "symmetric"}:
            gate_dim = self.hidden_dim * (2 if self.coupling_mode == "asymmetric" else 1)
            self.W_me = nn.Linear(self.hidden_dim * 3, gate_dim)
            self.W_ec = nn.Linear(self.hidden_dim * 3, gate_dim)
            self.W_mc = nn.Linear(self.hidden_dim * 3, gate_dim)
        self.trans_m = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.trans_e = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.trans_c = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.out_mech = nn.Linear(self.hidden_dim, 1)
        self.out_inter = nn.Linear(self.hidden_dim, 1)
        self.out_res = nn.Linear(self.hidden_dim, 1)

    @staticmethod
    def _first_bad(mask: torch.Tensor) -> tuple[int, ...]:
        return tuple(int(index) for index in torch.nonzero(mask, as_tuple=False)[0].tolist())

    def _validate_inputs(
        self,
        x_cont: torch.Tensor,
        x_cat: torch.Tensor,
        x_raw_phys: torch.Tensor,
    ) -> None:
        expected_shapes = (
            (x_cont, len(self.continuous_cols), "x_cont"),
            (x_cat, len(self.categorical_cols), "x_cat"),
            (x_raw_phys, 4, "x_raw_phys"),
        )
        batch_sizes = []
        for tensor, width, name in expected_shapes:
            if not isinstance(tensor, torch.Tensor) or tensor.ndim != 2 or tensor.shape[1] != width:
                raise ValueError(f"{name} must have shape (batch, {width})")
            batch_sizes.append(tensor.shape[0])
        if len(set(batch_sizes)) != 1:
            raise ValueError("x_cont, x_cat, and x_raw_phys must have matching batch sizes")
        if not x_cont.is_floating_point() or x_cont.is_complex():
            raise ValueError("Continuous inputs must be real floating-point values")
        invalid_cont = ~torch.isfinite(x_cont)
        if bool(invalid_cont.any()):
            row, column = self._first_bad(invalid_cont)
            raise ValueError(
                f"Invalid continuous value for '{self.continuous_cols[column]}' at batch row {row}: "
                f"{x_cont[row, column].item()!r}; expected a finite value"
            )
        if not x_raw_phys.is_floating_point() or x_raw_phys.is_complex():
            raise ValueError("Raw physiology inputs must be real floating-point values")
        invalid_phys = ~torch.isfinite(x_raw_phys)
        if bool(invalid_phys.any()):
            row, column = self._first_bad(invalid_phys)
            phys_cols = self.config["data"]["physiology_cols"]
            raise ValueError(
                f"Invalid physiology value for '{phys_cols[column]}' at batch row {row}: "
                f"{x_raw_phys[row, column].item()!r}; expected a finite value"
            )
        if x_cat.is_complex() or x_cat.dtype == torch.bool:
            raise ValueError("Categorical indices must be real integer values")
        invalid_cat = torch.zeros_like(x_cat, dtype=torch.bool)
        for column, name in enumerate(self.categorical_cols):
            values = x_cat[:, column]
            valid = torch.zeros_like(values, dtype=torch.bool)
            for code in self.category_values[name]:
                valid |= values == code
            invalid = ~valid
            if values.is_floating_point():
                invalid |= ~torch.isfinite(values) | (values != torch.trunc(values))
            invalid_cat[:, column] = invalid
        if bool(invalid_cat.any()):
            row, column = self._first_bad(invalid_cat)
            value = x_cat[row, column].item()
            allowed = list(self.category_values[self.categorical_cols[column]])
            raise ValueError(
                f"Invalid categorical index for '{self.categorical_cols[column]}' at batch row {row}: "
                f"{value!r}; expected one of {allowed}"
            )
        if any(max(values) >= self.embedding_cardinality for values in self.category_values.values()):
            raise ValueError("embedding_cardinality does not cover the configured categorical vocabularies")

    def _assemble(
        self,
        x_cont: torch.Tensor,
        x_cat: torch.Tensor,
        cont_indices: list[int],
        cat_indices: list[int],
    ) -> torch.Tensor:
        features = [x_cont[:, cont_indices]] if cont_indices else []
        for index in cat_indices:
            column = self.categorical_cols[index]
            features.append(self.embeddings[column](x_cat[:, index].long()))
        return torch.cat(features, dim=-1)

    def _couple(
        self,
        left: torch.Tensor,
        right: torch.Tensor,
        mismatch: torch.Tensor,
        gate_layer: nn.Linear,
        left_transform: nn.Linear,
        right_transform: nn.Linear,
    ) -> torch.Tensor:
        gate_logits = gate_layer(torch.cat((left, right, mismatch), dim=-1))
        forward_flow = left_transform(left) * right_transform(right)
        if self.coupling_mode == "symmetric":
            return torch.sigmoid(gate_logits) * forward_flow
        forward_gate, reverse_gate = torch.sigmoid(gate_logits).chunk(2, dim=-1)
        reverse_flow = right_transform(left) * left_transform(right)
        return forward_gate * forward_flow + reverse_gate * reverse_flow

    def forward(
        self,
        x_cont: torch.Tensor,
        x_cat: torch.Tensor,
        x_raw_phys: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        self._validate_inputs(x_cont, x_cat, x_raw_phys)
        age, max_hr, oldpeak, exang = (x_raw_phys[:, i:i + 1] for i in range(4))
        mismatch, _, _ = self.phys_module(age, max_hr, oldpeak, exang)
        mismatch_feature = self.dropout(F.relu(self.proj_me(mismatch)))
        h_m = self.dropout(F.relu(self.proj_metabolic(self._assemble(
            x_cont, x_cat, self.metabolic_cont_indices, self.metabolic_cat_indices
        ))))
        h_e = self.dropout(F.relu(self.proj_exercise(self._assemble(
            x_cont, x_cat, self.exercise_cont_indices, self.exercise_cat_indices
        ))))
        h_c = self.dropout(F.relu(self.proj_coronary(self._assemble(
            x_cont, x_cat, self.coronary_cont_indices, self.coronary_cat_indices
        ))))
        c_me = torch.zeros_like(h_m)
        c_ec = torch.zeros_like(h_e)
        c_mc = torch.zeros_like(h_c)
        if self.coupling_mode != "none":
            c_me = self._couple(h_m, h_e, mismatch_feature, self.W_me, self.trans_m, self.trans_e)
            c_ec = self._couple(h_e, h_c, mismatch_feature, self.W_ec, self.trans_e, self.trans_c)
            c_mc = self._couple(h_m, h_c, mismatch_feature, self.W_mc, self.trans_m, self.trans_c)
        z_mech = self.out_mech(mismatch_feature + h_c)
        z_inter = self.out_inter(c_me + c_ec + c_mc)
        z_res = self.out_res(h_m + h_e + h_c)
        z_total = z_mech + z_inter + z_res
        return {
            "z_total": z_total,
            "z_mech": z_mech,
            "z_inter": z_inter,
            "z_res": z_res,
            "m_e": mismatch,
        }

## 8. Model initialization and sanity checks

These are synthetic software checks only. They do not train on, evaluate, or report results for the uploaded clinical data.

In [ ]:
def run_model_sanity_checks() -> None:
    torch.manual_seed(101)
    model = PMRCNet(CONFIG).to(DEVICE)
    batch_size = 4
    x_cont = torch.randn(batch_size, len(CONFIG["data"]["continuous_cols"]), device=DEVICE)
    x_cat_columns = [
        torch.tensor(CONFIG["data"]["category_values"][name], device=DEVICE)
        for name in CONFIG["data"]["categorical_cols"]
    ]
    x_cat = torch.stack([
        column[torch.randint(len(column), (batch_size,), device=DEVICE)]
        for column in x_cat_columns
    ], dim=1)
    x_raw_phys = torch.tensor(
        [[55.0, 160.0, 1.5, 1.0], [40.0, 180.0, 0.0, 0.0],
         [62.0, 110.0, 3.0, 1.0], [50.0, 140.0, 1.0, 0.0]],
        dtype=torch.float32, device=DEVICE,
    )
    targets = torch.tensor([0.0, 1.0, 1.0, 0.0], device=DEVICE)
    outputs = model(x_cont, x_cat, x_raw_phys)
    assert outputs["z_total"].shape == (batch_size, 1)
    assert torch.allclose(outputs["z_total"], outputs["z_mech"] + outputs["z_inter"] + outputs["z_res"])
    probabilities = torch.sigmoid(outputs["z_total"])
    assert torch.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    loss = nn.BCEWithLogitsLoss()(outputs["z_total"].squeeze(-1), targets)
    loss.backward()
    assert torch.isfinite(loss)
    assert all(parameter.grad is None or torch.isfinite(parameter.grad).all() for parameter in model.parameters())

    with io.BytesIO() as buffer:
        torch.save(model.state_dict(), buffer)
        buffer.seek(0)
        restored = PMRCNet(CONFIG).to(DEVICE)
        restored.load_state_dict(torch.load(buffer, map_location=DEVICE, weights_only=True))
    restored.eval()
    model.eval()
    with torch.inference_mode():
        original_logit = model(x_cont, x_cat, x_raw_phys)["z_total"]
        restored_logit = restored(x_cont, x_cat, x_raw_phys)["z_total"]
    assert torch.allclose(original_logit, restored_logit)
    print("Synthetic software sanity checks passed: forward, additive logit, sigmoid range, backward, finite gradients, and state-dict round-trip.")
    print("These checks are not clinical experiment results.")

run_model_sanity_checks()


## 9. Experiment configuration

Quick mode is the default and is the only mode run unless explicitly changed. Nested mode keeps the v1.1 five-seed/five-outer/four-inner design fixed; max epochs, patience, min-delta, learning rate, batch size, and weight decay are configurable below. Use smaller epoch limits only for exploratory runtime checks, and report them as such.

In [ ]:
EXPERIMENT_MODE = "quick"  # Choose "quick" or "nested"; both are never run automatically.

TRAINING_CONFIG = {
    "batch_size": 8,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 100,
    "patience": 12,
    "min_delta": 1e-4,
    "gradient_clip_norm": 1.0,
    "num_workers": 0,
    "precision": "fp32",
}
NESTED_SEEDS = [21, 42, 123, 2026, 3407]
OUTER_FOLDS = 5
INNER_FOLDS = 4
THRESHOLD = 0.5
ECE_BINS = 10

if EXPERIMENT_MODE not in {"quick", "nested"}:
    raise ValueError("EXPERIMENT_MODE must be 'quick' or 'nested'.")
if TRAINING_CONFIG["precision"] != "fp32":
    raise ValueError("This v1.1 notebook supports FP32 only.")
if TRAINING_CONFIG["batch_size"] < 1 or TRAINING_CONFIG["max_epochs"] < 1:
    raise ValueError("batch_size and max_epochs must be positive.")
if EXPERIMENT_MODE == "nested" and (len(NESTED_SEEDS) != 5 or len(set(NESTED_SEEDS)) != 5):
    raise ValueError("Research mode requires the five distinct configured v1.1 seeds.")

RESULTS_ROOT = Path.cwd() / "PMRC_Net_Colab_Results"
EXPERIMENT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
EXPERIMENT_DIR = RESULTS_ROOT / EXPERIMENT_ID
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
FIGURES_DIR = EXPERIMENT_DIR / "figures"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Mode: {EXPERIMENT_MODE}; experiment ID: {EXPERIMENT_ID}")
print(f"Output directory: {EXPERIMENT_DIR}")


## 10. Quick experiment / nested evaluation selection

Review EXPERIMENT_MODE above. Quick mode uses a stratified 64% training, 16% validation, and 20% untouched test partition (the validation fraction is 20% of the 80% train/validation partition). Research mode runs the complete configured nested evaluation.

## 11. Model training

Training helpers use v1.1 BCE-with-logits and AdamW. Scalers are fitted only on each training partition. Validation drives early stopping only in quick mode and inner folds only in nested mode. An outer test partition is evaluated after a fresh refit for the inner-selected epoch count.

In [ ]:
def set_random_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def build_tensor_dataset(
    indices: np.ndarray,
    scaler: StandardScaler | None = None,
    fit_scaler: bool = False,
) -> tuple[TensorDataset, StandardScaler]:
    indices = np.asarray(indices, dtype=np.int64)
    if indices.ndim != 1 or indices.size == 0:
        raise ValueError("A non-empty one-dimensional row-index array is required.")
    frame = DATAFRAME.iloc[indices]
    continuous = CONFIG["data"]["continuous_cols"]
    if fit_scaler:
        if scaler is None:
            scaler = StandardScaler()
        continuous_values = scaler.fit_transform(frame[continuous].to_numpy(dtype=np.float64)).astype(np.float32)
    else:
        if scaler is None:
            raise ValueError("A scaler fitted on the training partition is required for transform-only preprocessing.")
        continuous_values = scaler.transform(frame[continuous].to_numpy(dtype=np.float64)).astype(np.float32)

    categorical_values = frame[CONFIG["data"]["categorical_cols"]].to_numpy(dtype=np.int64)
    physiology_values = frame[CONFIG["data"]["physiology_cols"]].to_numpy(dtype=np.float32)
    labels = frame[CONFIG["data"]["target"]].to_numpy(dtype=np.float32)
    row_indices = frame["row_index"].to_numpy(dtype=np.int64)
    dataset = TensorDataset(
        torch.from_numpy(continuous_values),
        torch.from_numpy(categorical_values),
        torch.from_numpy(physiology_values),
        torch.from_numpy(labels),
        torch.from_numpy(row_indices),
    )
    return dataset, scaler


def make_loader(
    indices: np.ndarray,
    scaler: StandardScaler | None = None,
    fit_scaler: bool = False,
    shuffle: bool = False,
    seed: int = 0,
) -> tuple[DataLoader, StandardScaler]:
    dataset, scaler = build_tensor_dataset(indices, scaler=scaler, fit_scaler=fit_scaler)
    generator = torch.Generator().manual_seed(int(seed))
    loader = DataLoader(
        dataset,
        batch_size=int(TRAINING_CONFIG["batch_size"]),
        shuffle=shuffle,
        num_workers=int(TRAINING_CONFIG["num_workers"]),
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator if shuffle else None,
    )
    return loader, scaler


def fit_model(
    train_loader: DataLoader,
    validation_loader: DataLoader | None,
    max_epochs: int,
    patience: int,
    seed: int,
) -> tuple[PMRCNet, list[dict], int]:
    set_random_seed(seed)
    model = PMRCNet(CONFIG).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(TRAINING_CONFIG["learning_rate"]),
        weight_decay=float(TRAINING_CONFIG["weight_decay"]),
    )
    objective = nn.BCEWithLogitsLoss()
    history = []
    best_validation_loss = float("inf")
    best_epoch = 0
    best_state = None
    wait = 0

    try:
        for epoch in range(1, int(max_epochs) + 1):
            model.train()
            train_total = 0.0
            train_count = 0
            for batch in train_loader:
                x_cont, x_cat, x_phys, labels = [value.to(DEVICE, non_blocking=True) for value in batch[:4]]
                optimizer.zero_grad(set_to_none=True)
                logits = model(x_cont, x_cat, x_phys)["z_total"].squeeze(-1)
                loss = objective(logits, labels)
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError(f"Non-finite training loss at epoch {epoch}.")
                loss.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), float(TRAINING_CONFIG["gradient_clip_norm"])
                )
                if not bool(torch.isfinite(gradient_norm)):
                    raise FloatingPointError(f"Non-finite gradient norm at epoch {epoch}.")
                optimizer.step()
                count = int(labels.shape[0])
                train_total += float(loss.detach()) * count
                train_count += count
            train_loss = train_total / train_count
            validation_loss = None

            if validation_loader is not None:
                model.eval()
                valid_total = 0.0
                valid_count = 0
                with torch.inference_mode():
                    for batch in validation_loader:
                        x_cont, x_cat, x_phys, labels = [value.to(DEVICE, non_blocking=True) for value in batch[:4]]
                        logits = model(x_cont, x_cat, x_phys)["z_total"].squeeze(-1)
                        loss = objective(logits, labels)
                        if not bool(torch.isfinite(loss)):
                            raise FloatingPointError(f"Non-finite validation loss at epoch {epoch}.")
                        count = int(labels.shape[0])
                        valid_total += float(loss) * count
                        valid_count += count
                validation_loss = valid_total / valid_count
                if validation_loss < best_validation_loss - float(TRAINING_CONFIG["min_delta"]):
                    best_validation_loss = validation_loss
                    best_epoch = epoch
                    best_state = {
                        name: value.detach().cpu().clone()
                        for name, value in model.state_dict().items()
                    }
                    wait = 0
                else:
                    wait += 1
            history.append({
                "epoch": epoch,
                "train_loss": float(train_loss),
                "validation_loss": None if validation_loss is None else float(validation_loss),
            })
            if validation_loader is not None and wait >= int(patience):
                break

        if validation_loader is not None:
            if best_state is None:
                raise RuntimeError("Training completed without a finite validation checkpoint.")
            model.load_state_dict(best_state)
        else:
            best_epoch = int(max_epochs)
    except torch.cuda.OutOfMemoryError as exc:
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        raise RuntimeError(
            "CUDA ran out of memory. Reduce TRAINING_CONFIG['batch_size'] or use a CPU runtime, then rerun with a new experiment ID."
        ) from exc
    finally:
        del optimizer
    return model, history, int(best_epoch)


def predict_loader(model: PMRCNet, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    probabilities, labels, row_indices = [], [], []
    with torch.inference_mode():
        for batch in loader:
            x_cont, x_cat, x_phys, targets, indices = [value.to(DEVICE, non_blocking=True) for value in batch]
            logits = model(x_cont, x_cat, x_phys)["z_total"]
            probabilities.extend(torch.sigmoid(logits).squeeze(-1).cpu().numpy().tolist())
            labels.extend(targets.cpu().numpy().astype(int).tolist())
            row_indices.extend(indices.cpu().numpy().astype(int).tolist())
    return (
        np.asarray(row_indices, dtype=np.int64),
        np.asarray(labels, dtype=np.int64),
        np.asarray(probabilities, dtype=np.float64),
    )


def make_prediction_loader(indices: np.ndarray, scaler: StandardScaler) -> DataLoader:
    dataset, _ = build_tensor_dataset(indices, scaler=scaler, fit_scaler=False)
    return DataLoader(
        dataset,
        batch_size=int(TRAINING_CONFIG["batch_size"]),
        shuffle=False,
        num_workers=int(TRAINING_CONFIG["num_workers"]),
        pin_memory=(DEVICE.type == "cuda"),
    )


## 12. Evaluation

Quick evaluation reports only the untouched test partition. Nested evaluation verifies one outer-fold prediction per row per seed, with labels and fold assignments aligned before calculating metrics. The nested primary point estimates are computed from each seed's pooled outer OOF predictions; mean and sample SD are then reported across the five seeds.

In [ ]:
def expected_calibration_error(labels: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> float:
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    edges = np.linspace(0.0, 1.0, int(bins) + 1)
    error = 0.0
    for index in range(int(bins)):
        lower, upper = edges[index], edges[index + 1]
        in_bin = (probabilities >= lower) & (
            (probabilities <= upper) if index == bins - 1 else (probabilities < upper)
        )
        if in_bin.any():
            error += float(in_bin.mean()) * abs(float(labels[in_bin].mean()) - float(probabilities[in_bin].mean()))
    return float(error)


def calculate_metrics(labels: np.ndarray, probabilities: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if labels.ndim != 1 or probabilities.ndim != 1 or labels.shape != probabilities.shape:
        raise ValueError("Labels and probabilities must be aligned one-dimensional arrays.")
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Predicted probabilities must be finite values in [0, 1].")
    if not np.isin(labels, [0, 1]).all() or np.unique(labels).size != 2:
        raise ValueError("Metrics require both binary target classes.")
    predicted = (probabilities >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "threshold": float(THRESHOLD),
        "accuracy": float(accuracy_score(labels, predicted)),
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "sensitivity": float(recall_score(labels, predicted, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if tn + fp else 0.0,
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "mcc": float(matthews_corrcoef(labels, predicted)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "brier_score": float(brier_score_loss(labels, probabilities)),
        "ece": expected_calibration_error(labels, probabilities, ECE_BINS),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


METRIC_NAMES = [
    "accuracy", "precision", "sensitivity", "specificity", "f1", "mcc",
    "roc_auc", "average_precision", "brier_score", "ece",
]


def run_quick_experiment() -> dict:
    seed = int(CONFIG["seed"])
    set_random_seed(seed)
    row_indices = np.arange(len(DATAFRAME), dtype=np.int64)
    labels = DATAFRAME[CONFIG["data"]["target"]].to_numpy(dtype=int)
    train_validation_indices, test_indices = train_test_split(
        row_indices, test_size=0.20, random_state=seed, stratify=labels
    )
    train_indices, validation_indices = train_test_split(
        train_validation_indices, test_size=0.20, random_state=seed, stratify=labels[train_validation_indices]
    )
    if set(train_indices) & set(validation_indices) or set(train_indices) & set(test_indices) or set(validation_indices) & set(test_indices):
        raise RuntimeError("Quick train, validation, and test partitions must be disjoint.")

    train_loader, scaler = make_loader(
        train_indices, fit_scaler=True, shuffle=True, seed=seed
    )
    validation_loader, _ = make_loader(
        validation_indices, scaler=scaler, shuffle=False, seed=seed
    )
    model, history, best_epoch = fit_model(
        train_loader, validation_loader,
        max_epochs=TRAINING_CONFIG["max_epochs"],
        patience=TRAINING_CONFIG["patience"],
        seed=seed,
    )

    checkpoint_path = CHECKPOINT_DIR / "quick_model.pt"
    torch.save({
        "version": "1.1",
        "experiment_id": EXPERIMENT_ID,
        "mode": "quick",
        "model_state": {name: value.detach().cpu() for name, value in model.state_dict().items()},
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
        "best_epoch": best_epoch,
        "training_config": TRAINING_CONFIG,
        "row_indices_used_for_training": DATAFRAME.iloc[train_indices]["row_index"].tolist(),
    }, checkpoint_path)

    test_loader = make_prediction_loader(test_indices, scaler)
    test_rows, test_labels, test_probabilities = predict_loader(model, test_loader)
    predictions = pd.DataFrame({
        "row_index": test_rows,
        "target": test_labels,
        "probability": test_probabilities,
        "partition": "untouched_test",
    }).sort_values("row_index").reset_index(drop=True)
    metrics = calculate_metrics(test_labels, test_probabilities)
    return {
        "mode": "quick",
        "model": model,
        "scaler": scaler,
        "checkpoint_path": checkpoint_path,
        "history": {"fit": history, "best_epoch": best_epoch},
        "predictions": predictions,
        "plot_predictions": predictions.copy(),
        "metrics": metrics,
        "metrics_records": [{"record_type": "quick_test", **metrics}],
        "fold_metrics": [],
        "seed_metrics": [],
        "training_indices": train_indices.tolist(),
        "validation_indices": validation_indices.tolist(),
        "test_indices": test_indices.tolist(),
    }


def validate_oof_predictions(
    frame: pd.DataFrame,
    seed: int,
    expected_fold_ids: np.ndarray,
) -> None:
    ordered = frame.sort_values("row_index").reset_index(drop=True)
    expected_rows = DATAFRAME["row_index"].to_numpy(dtype=np.int64)
    expected_labels = DATAFRAME[CONFIG["data"]["target"]].to_numpy(dtype=np.int64)
    if len(ordered) != len(DATAFRAME) or ordered["row_index"].duplicated().any():
        raise ValueError(f"Seed {seed} OOF predictions must cover each deduplicated row exactly once.")
    if not np.array_equal(ordered["row_index"].to_numpy(dtype=np.int64), expected_rows):
        raise ValueError(f"Seed {seed} OOF row-index alignment failed.")
    if not np.array_equal(ordered["target"].to_numpy(dtype=np.int64), expected_labels):
        raise ValueError(f"Seed {seed} OOF labels do not match the cleaned dataset.")
    if not np.array_equal(ordered["outer_fold"].to_numpy(dtype=np.int64), expected_fold_ids):
        raise ValueError(f"Seed {seed} outer-fold assignments do not match the generated split manifest.")
    probabilities = ordered["probability"].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError(f"Seed {seed} OOF probabilities are not finite values in [0, 1].")


def run_nested_experiment() -> dict:
    if len(NESTED_SEEDS) != 5 or OUTER_FOLDS != 5 or INNER_FOLDS != 4:
        raise ValueError("Research evaluation must use five seeds, five outer folds, and four inner folds.")
    labels = DATAFRAME[CONFIG["data"]["target"]].to_numpy(dtype=int)
    all_fold_frames = []
    fold_metric_records = []
    seed_metric_records = []
    inner_history_records = []
    refit_history_records = []
    selected_checkpoint = None

    for seed in NESTED_SEEDS:
        outer_splitter = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=int(seed))
        seed_frames = []
        expected_fold_ids = np.zeros(len(DATAFRAME), dtype=np.int64)
        for outer_fold, (outer_train, outer_test) in enumerate(
            outer_splitter.split(np.zeros(len(DATAFRAME)), labels), start=1
        ):
            expected_fold_ids[outer_test] = outer_fold
            if set(outer_train) & set(outer_test):
                raise RuntimeError("Outer training and test rows overlap.")
            inner_splitter = StratifiedKFold(
                n_splits=INNER_FOLDS, shuffle=True,
                random_state=int(seed) + 1009 * outer_fold,
            )
            inner_best_epochs = []
            inner_records = []
            outer_train_labels = labels[outer_train]

            print(f"Seed {seed}: outer fold {outer_fold}/{OUTER_FOLDS} — selecting training duration on inner folds.")
            for inner_fold, (inner_train_local, inner_valid_local) in enumerate(
                inner_splitter.split(np.zeros(len(outer_train)), outer_train_labels), start=1
            ):
                inner_train = outer_train[inner_train_local]
                inner_valid = outer_train[inner_valid_local]
                if set(inner_train) & set(inner_valid) or set(inner_train) & set(outer_test) or set(inner_valid) & set(outer_test):
                    raise RuntimeError("Inner partitions must be disjoint and exclude the outer test rows.")
                inner_seed = int(seed) * 100000 + outer_fold * 1000 + inner_fold
                inner_train_loader, inner_scaler = make_loader(
                    inner_train, fit_scaler=True, shuffle=True, seed=inner_seed
                )
                inner_valid_loader, _ = make_loader(
                    inner_valid, scaler=inner_scaler, shuffle=False, seed=inner_seed
                )
                inner_model, inner_history, best_epoch = fit_model(
                    inner_train_loader, inner_valid_loader,
                    max_epochs=TRAINING_CONFIG["max_epochs"],
                    patience=TRAINING_CONFIG["patience"],
                    seed=inner_seed,
                )
                inner_best_epochs.append(best_epoch)
                inner_records.append({
                    "inner_fold": inner_fold,
                    "best_epoch": best_epoch,
                    "history": inner_history,
                })
                inner_history_records.append({
                    "seed": int(seed), "outer_fold": outer_fold,
                    "inner_fold": inner_fold, "history": inner_history,
                })
                del inner_model, inner_train_loader, inner_valid_loader, inner_scaler
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()

            selected_epochs = max(1, int(math.ceil(float(np.median(inner_best_epochs)))))
            refit_seed = int(seed) * 100000 + outer_fold * 1000 + 999
            outer_train_loader, outer_scaler = make_loader(
                outer_train, fit_scaler=True, shuffle=True, seed=refit_seed
            )
            outer_model, refit_history, _ = fit_model(
                outer_train_loader, None,
                max_epochs=selected_epochs,
                patience=selected_epochs,
                seed=refit_seed,
            )
            refit_history_records.append({
                "seed": int(seed), "outer_fold": outer_fold,
                "selected_epochs": selected_epochs, "history": refit_history,
            })

            outer_test_loader = make_prediction_loader(outer_test, outer_scaler)
            test_rows, test_labels, test_probabilities = predict_loader(outer_model, outer_test_loader)
            fold_frame = pd.DataFrame({
                "row_index": test_rows,
                "seed": int(seed),
                "outer_fold": outer_fold,
                "target": test_labels,
                "probability": test_probabilities,
            }).sort_values("row_index").reset_index(drop=True)
            fold_metrics = calculate_metrics(test_labels, test_probabilities)
            fold_metric_records.append({
                "record_type": "outer_fold", "seed": int(seed),
                "outer_fold": outer_fold, "selected_epochs": selected_epochs,
                **fold_metrics,
            })
            seed_frames.append(fold_frame)

            checkpoint_path = CHECKPOINT_DIR / f"seed_{seed}_outer_fold_{outer_fold}.pt"
            torch.save({
                "version": "1.1",
                "experiment_id": EXPERIMENT_ID,
                "mode": "nested",
                "seed": int(seed),
                "outer_fold": outer_fold,
                "selected_epochs": selected_epochs,
                "inner_best_epochs": inner_best_epochs,
                "outer_test_used_for_selection": False,
                "model_state": {name: value.detach().cpu() for name, value in outer_model.state_dict().items()},
                "scaler_mean": outer_scaler.mean_.tolist(),
                "scaler_scale": outer_scaler.scale_.tolist(),
                "row_indices_used_for_training": DATAFRAME.iloc[outer_train]["row_index"].tolist(),
            }, checkpoint_path)
            if seed == 42 and outer_fold == 1:
                selected_checkpoint = checkpoint_path
            del outer_model, outer_train_loader, outer_test_loader, outer_scaler
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        seed_frame = pd.concat(seed_frames, ignore_index=True).sort_values("row_index").reset_index(drop=True)
        validate_oof_predictions(seed_frame, int(seed), expected_fold_ids)
        seed_metrics = calculate_metrics(
            seed_frame["target"].to_numpy(dtype=int),
            seed_frame["probability"].to_numpy(dtype=float),
        )
        seed_metric_records.append({"record_type": "seed_oof", "seed": int(seed), **seed_metrics})
        all_fold_frames.append(seed_frame)

    all_predictions = pd.concat(all_fold_frames, ignore_index=True)
    seed42_predictions = all_predictions[all_predictions["seed"] == 42].copy()
    if selected_checkpoint is None or seed42_predictions.empty:
        raise RuntimeError("Could not locate seed-42 OOF predictions and checkpoint for the primary nested plots.")

    metric_records = list(fold_metric_records) + list(seed_metric_records)
    for metric_name in METRIC_NAMES:
        values = [record[metric_name] for record in seed_metric_records]
        metric_records.append({
            "record_type": "mean_across_seeds",
            "seed": None,
            metric_name: float(np.mean(values)),
        })
        metric_records.append({
            "record_type": "sample_sd_across_seeds",
            "seed": None,
            metric_name: float(np.std(values, ddof=1)),
        })

    return {
        "mode": "nested",
        "model": None,
        "scaler": None,
        "checkpoint_path": selected_checkpoint,
        "checkpoint_paths": sorted(CHECKPOINT_DIR.glob("seed_*_outer_fold_*.pt")),
        "history": {
            "inner_runs": inner_history_records,
            "outer_refits": refit_history_records,
        },
        "predictions": all_predictions,
        "plot_predictions": seed42_predictions.sort_values("row_index").reset_index(drop=True),
        "metrics": seed_metric_records,
        "metrics_records": metric_records,
        "fold_metrics": fold_metric_records,
        "seed_metrics": seed_metric_records,
    }


if EXPERIMENT_MODE == "quick":
    RESULTS = run_quick_experiment()
else:
    RESULTS = run_nested_experiment()

print(f"Evaluation completed for mode={RESULTS['mode']}.")
print("No historical experiment artifacts were loaded, modified, or retrained.")


## 13. Calibration and uncertainty analysis

Calibration is computed only from predictions made by the trained selected model on the quick untouched test set, or from verified outer-fold OOF probabilities in nested mode (seed 42 for the displayed curve). The labels are aligned with those predictions. No fresh untrained model is initialized here, no scaler is fit on evaluation data, and no post-hoc calibration or threshold selection is performed. ECE uses ten equal-width bins, with the final bin including probability 1.0. Calibration and uncertainty estimates are descriptive and bin-dependent; no patient-level interval is possible without verified patient identifiers.

In [ ]:
def calibration_table(labels: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> pd.DataFrame:
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    edges = np.linspace(0.0, 1.0, int(bins) + 1)
    rows = []
    for index in range(int(bins)):
        lower, upper = float(edges[index]), float(edges[index + 1])
        mask = (probabilities >= lower) & (
            (probabilities <= upper) if index == bins - 1 else (probabilities < upper)
        )
        count = int(mask.sum())
        rows.append({
            "bin": index + 1,
            "lower": lower,
            "upper": upper,
            "count": count,
            "mean_probability": float(probabilities[mask].mean()) if count else np.nan,
            "observed_fraction_positive": float(labels[mask].mean()) if count else np.nan,
            "absolute_calibration_gap": (
                abs(float(labels[mask].mean()) - float(probabilities[mask].mean())) if count else np.nan
            ),
        })
    return pd.DataFrame(rows)


PLOT_LABELS = RESULTS["plot_predictions"]["target"].to_numpy(dtype=int)
PLOT_PROBABILITIES = RESULTS["plot_predictions"]["probability"].to_numpy(dtype=float)
CALIBRATION_TABLE = calibration_table(PLOT_LABELS, PLOT_PROBABILITIES, ECE_BINS)
PLOT_CALIBRATION_METRICS = {
    "brier_score": float(brier_score_loss(PLOT_LABELS, PLOT_PROBABILITIES)),
    "ece": expected_calibration_error(PLOT_LABELS, PLOT_PROBABILITIES, ECE_BINS),
    "ece_bins": ECE_BINS,
    "threshold": THRESHOLD,
    "prediction_source": "quick untouched test" if RESULTS["mode"] == "quick" else "verified seed-42 outer-fold OOF",
}

if RESULTS["mode"] == "nested":
    CALIBRATION_BY_SEED = pd.DataFrame([
        {
            "seed": int(seed),
            "brier_score": float(record["brier_score"]),
            "ece": float(record["ece"]),
        }
        for seed, record in zip(NESTED_SEEDS, RESULTS["seed_metrics"])
    ])
    CALIBRATION_SUMMARY = {
        metric: {
            "mean": float(CALIBRATION_BY_SEED[metric].mean()),
            "sample_sd": float(CALIBRATION_BY_SEED[metric].std(ddof=1)),
        }
        for metric in ("brier_score", "ece")
    }
else:
    CALIBRATION_BY_SEED = pd.DataFrame([{
        "seed": int(CONFIG["seed"]),
        "brier_score": PLOT_CALIBRATION_METRICS["brier_score"],
        "ece": PLOT_CALIBRATION_METRICS["ece"],
    }])
    CALIBRATION_SUMMARY = {"note": "One held-out test set; no across-seed uncertainty estimate."}

print("Calibration data source:", PLOT_CALIBRATION_METRICS["prediction_source"])
print(json.dumps(PLOT_CALIBRATION_METRICS, indent=2))
display(CALIBRATION_TABLE)


## 14. Visualizations

All curves below use stored model probabilities and corresponding ground-truth labels. Nested ROC, precision-recall, calibration, and confusion-matrix plots use seed-42 outer OOF predictions. Quick plots use the untouched test predictions. The loss plot uses actual training history; nested loss curves aggregate observed inner-fold histories.

In [ ]:
def save_experiment_figures(results: dict) -> dict:
    prediction_frame = results["plot_predictions"].sort_values("row_index").reset_index(drop=True)
    labels = prediction_frame["target"].to_numpy(dtype=int)
    probabilities = prediction_frame["probability"].to_numpy(dtype=float)
    paths = {}

    if results["mode"] == "quick":
        history_runs = [results["history"]["fit"]]
        loss_title = "Quick-mode training and validation loss"
    else:
        history_runs = [record["history"] for record in results["history"]["inner_runs"]]
        loss_title = "Nested-mode mean inner-fold training and validation loss"

    train_by_epoch, valid_by_epoch = {}, {}
    for run in history_runs:
        for row in run:
            epoch = int(row["epoch"])
            train_by_epoch.setdefault(epoch, []).append(float(row["train_loss"]))
            if row["validation_loss"] is not None:
                valid_by_epoch.setdefault(epoch, []).append(float(row["validation_loss"]))
    epochs = sorted(train_by_epoch)
    train_means = [float(np.mean(train_by_epoch[epoch])) for epoch in epochs]
    valid_epochs = sorted(valid_by_epoch)
    valid_means = [float(np.mean(valid_by_epoch[epoch])) for epoch in valid_epochs]
    fig, axis = plt.subplots(figsize=(7, 4.5))
    axis.plot(epochs, train_means, label="Training loss")
    if valid_epochs:
        axis.plot(valid_epochs, valid_means, label="Validation loss")
    axis.set(title=loss_title, xlabel="Epoch", ylabel="BCE loss")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    paths["loss"] = FIGURES_DIR / "training_validation_loss.png"
    fig.savefig(paths["loss"], dpi=160)
    plt.show()
    plt.close(fig)

    fpr, tpr, _ = roc_curve(labels, probabilities)
    roc_auc = float(roc_auc_score(labels, probabilities))
    fig, axis = plt.subplots(figsize=(5.5, 5))
    axis.plot(fpr, tpr, label=f"Model (ROC-AUC={roc_auc:.3f})")
    axis.plot([0, 1], [0, 1], linestyle="--", color="gray")
    axis.set(title="ROC curve", xlabel="False positive rate", ylabel="Sensitivity")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    paths["roc"] = FIGURES_DIR / "roc_curve.png"
    fig.savefig(paths["roc"], dpi=160)
    plt.show()
    plt.close(fig)

    precision_values, recall_values, _ = precision_recall_curve(labels, probabilities)
    average_precision = float(average_precision_score(labels, probabilities))
    fig, axis = plt.subplots(figsize=(5.5, 5))
    axis.plot(recall_values, precision_values, label=f"Model (AP={average_precision:.3f})")
    axis.set(title="Precision-recall curve", xlabel="Recall", ylabel="Precision")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    paths["precision_recall"] = FIGURES_DIR / "precision_recall_curve.png"
    fig.savefig(paths["precision_recall"], dpi=160)
    plt.show()
    plt.close(fig)

    bin_table = calibration_table(labels, probabilities, ECE_BINS)
    nonempty = bin_table[bin_table["count"] > 0]
    fig, axis = plt.subplots(figsize=(5.5, 5))
    axis.plot(
        nonempty["mean_probability"], nonempty["observed_fraction_positive"],
        marker="o", label=f"Model (ECE={expected_calibration_error(labels, probabilities, ECE_BINS):.3f})",
    )
    axis.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Ideal")
    axis.set(title="Calibration curve", xlabel="Mean predicted probability", ylabel="Observed fraction positive")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    paths["calibration"] = FIGURES_DIR / "calibration_curve.png"
    fig.savefig(paths["calibration"], dpi=160)
    plt.show()
    plt.close(fig)

    matrix = confusion_matrix(labels, (probabilities >= THRESHOLD).astype(int), labels=[0, 1])
    fig, axis = plt.subplots(figsize=(5, 4.5))
    image = axis.imshow(matrix, cmap="Blues")
    axis.set(
        title=f"Confusion matrix (threshold={THRESHOLD:.2f})",
        xlabel="Predicted label", ylabel="Observed label",
        xticks=[0, 1], yticks=[0, 1],
        xticklabels=["0", "1"], yticklabels=["0", "1"],
    )
    for (row, column), value in np.ndenumerate(matrix):
        axis.text(column, row, str(int(value)), ha="center", va="center")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.tight_layout()
    paths["confusion_matrix"] = FIGURES_DIR / "confusion_matrix.png"
    fig.savefig(paths["confusion_matrix"], dpi=160)
    plt.show()
    plt.close(fig)
    return paths


FIGURE_PATHS = save_experiment_figures(RESULTS)


## 15. Results summary

Quick metrics summarize the untouched test partition only. Nested summaries show each seed's pooled outer OOF metrics and the mean plus sample SD across seeds. Average precision is reported as scikit-learn average precision, not trapezoidal PR-AUC.

In [ ]:
if RESULTS["mode"] == "quick":
    print("Quick experiment — untouched test set")
    display(pd.DataFrame([RESULTS["metrics"]]).T.rename(columns={0: "value"}))
    print(f"Training stopped at epoch {RESULTS['history']['best_epoch']} selected by validation BCE.")
else:
    print("Nested experiment — pooled outer OOF metrics by seed")
    display(pd.DataFrame(RESULTS["seed_metrics"]))
    summary_rows = []
    for metric_name in METRIC_NAMES:
        values = [record[metric_name] for record in RESULTS["seed_metrics"]]
        summary_rows.append({
            "metric": metric_name,
            "mean_across_seeds": float(np.mean(values)),
            "sample_sd_across_seeds": float(np.std(values, ddof=1)),
        })
    NESTED_SUMMARY = pd.DataFrame(summary_rows)
    display(NESTED_SUMMARY)

print("Classification threshold:", THRESHOLD)
print("Prediction count used for the displayed curves:", len(RESULTS["plot_predictions"]))
print("Analysis unit: deduplicated data row; patient independence is unverified.")


## 16. Checkpoint saving and result export

Write a unique run directory containing the model checkpoint(s), configuration, training history, metrics, predictions/OOF predictions, evaluation report, calibration results, confusion matrix, and figures. The notebook does not overwrite historical v1.0 or v1.1 project artifacts.

In [ ]:
def json_default(value):
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if torch.is_tensor(value):
        return value.detach().cpu().tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__} to JSON.")


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, default=json_default), encoding="utf-8")


metrics_frame = pd.DataFrame(RESULTS["metrics_records"])
metrics_frame.to_csv(EXPERIMENT_DIR / "metrics.csv", index=False)
if RESULTS["mode"] == "quick":
    RESULTS["predictions"].to_csv(EXPERIMENT_DIR / "test_predictions.csv", index=False)
else:
    RESULTS["predictions"].to_csv(EXPERIMENT_DIR / "oof_predictions.csv", index=False)

write_json(EXPERIMENT_DIR / "training_history.json", RESULTS["history"])
write_json(EXPERIMENT_DIR / "calibration_results.json", {
    "displayed_curve_metrics": PLOT_CALIBRATION_METRICS,
    "across_seed_summary": CALIBRATION_SUMMARY,
})
calibration_export = CALIBRATION_TABLE.copy()
calibration_export.insert(0, "prediction_source", PLOT_CALIBRATION_METRICS["prediction_source"])
calibration_export.to_csv(EXPERIMENT_DIR / "calibration_bins.csv", index=False)
CALIBRATION_BY_SEED.to_csv(EXPERIMENT_DIR / "calibration_by_seed.csv", index=False)

plot_frame = RESULTS["plot_predictions"].sort_values("row_index").reset_index(drop=True)
confusion_values = confusion_matrix(
    plot_frame["target"].to_numpy(dtype=int),
    (plot_frame["probability"].to_numpy(dtype=float) >= THRESHOLD).astype(int),
    labels=[0, 1],
)
pd.DataFrame(
    confusion_values,
    index=pd.Index([0, 1], name="observed"),
    columns=pd.Index([0, 1], name="predicted"),
).to_csv(EXPERIMENT_DIR / "confusion_matrix.csv")

runtime_metadata = {
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "torch_version": str(torch.__version__),
    "numpy_version": str(np.__version__),
    "pandas_version": str(pd.__version__),
}
experiment_config = {
    "version": "PMRC-Net v1.1",
    "experiment_id": EXPERIMENT_ID,
    "mode": EXPERIMENT_MODE,
    "dataset_filename": UPLOADED_FILENAME,
    "dataset_sha256": DATA_SHA256,
    "dataset_audit": DATA_AUDIT,
    "configuration": CONFIG,
    "training": TRAINING_CONFIG,
    "nested_seeds": NESTED_SEEDS if EXPERIMENT_MODE == "nested" else [],
    "outer_folds": OUTER_FOLDS if EXPERIMENT_MODE == "nested" else None,
    "inner_folds": INNER_FOLDS if EXPERIMENT_MODE == "nested" else None,
    "threshold": THRESHOLD,
    "ece_bins": ECE_BINS,
    "precision": "FP32",
    "runtime": runtime_metadata,
}
write_json(EXPERIMENT_DIR / "configuration.json", experiment_config)

if RESULTS["mode"] == "nested":
    pd.DataFrame(RESULTS["fold_metrics"]).to_csv(EXPERIMENT_DIR / "per_fold_metrics.csv", index=False)
    pd.DataFrame(RESULTS["seed_metrics"]).to_csv(EXPERIMENT_DIR / "per_seed_metrics.csv", index=False)

if RESULTS["checkpoint_path"] is None or not Path(RESULTS["checkpoint_path"]).is_file():
    raise FileNotFoundError("The selected trained checkpoint was not saved.")
saved_checkpoint = torch.load(RESULTS["checkpoint_path"], map_location="cpu", weights_only=True)
checkpoint_test_model = PMRCNet(CONFIG).cpu()
checkpoint_test_model.load_state_dict(saved_checkpoint["model_state"])
if len(RESULTS["plot_predictions"]) == 0:
    raise RuntimeError("No held-out or OOF predictions are available for the report.")
del checkpoint_test_model, saved_checkpoint

if RESULTS["mode"] == "quick":
    report_metrics = RESULTS["metrics"]
    metric_lines = "\n".join(f"- {name}: {value:.6f}" for name, value in report_metrics.items())
else:
    metric_lines = "\n".join(
        f"- {row.metric}: {row.mean_across_seeds:.6f} mean, {row.sample_sd_across_seeds:.6f} sample SD"
        for row in NESTED_SUMMARY.itertuples(index=False)
    )
evaluation_report = f"""# PMRC-Net Colab Evaluation Report

- Experiment ID: {EXPERIMENT_ID}
- Model version: PMRC-Net v1.1
- Evaluation mode: {EXPERIMENT_MODE}
- Uploaded file: {UPLOADED_FILENAME}
- Dataset SHA-256: {DATA_SHA256}
- Raw rows: {DATA_AUDIT['raw_rows']}
- Exact duplicate raw rows: {DATA_AUDIT['raw_exact_duplicate_rows']}
- Exact-row-deduplicated observations: {DATA_AUDIT['deduplicated_rows']}
- Device: {runtime_metadata['device']} ({runtime_metadata['gpu_name'] or 'CPU'})
- Precision: FP32
- Threshold: {THRESHOLD}

## Metrics

{metric_lines}

## Method and limitations

Quick mode uses stratified train/validation/test partitions; only the training partition fits the scaler, validation selects the early-stopping checkpoint, and the test set is evaluated once afterward. Nested mode uses five seeds, five outer folds, and four inner folds; inner validation BCE selects the median best epoch count and a fresh model is refit on all outer-training rows before outer-test prediction. Outer test rows are excluded from preprocessing and selection.

Predictions are row-level, not patient-level. The established merged reference has 602 exact-row-deduplicated observations, but patient independence is unverified. This is internal validation only: no independent external validation, causal physiological relationship, clinical readiness, or model superiority is established. Calibration is descriptive and uses held-out predictions; ECE depends on its binning method. Average precision is not trapezoidal PR-AUC.
"""
(EXPERIMENT_DIR / "evaluation_report.md").write_text(evaluation_report, encoding="utf-8")

zip_base = RESULTS_ROOT / f"PMRC_Net_Colab_Results_{EXPERIMENT_ID}"
RESULTS_ZIP = Path(shutil.make_archive(str(zip_base), "zip", root_dir=RESULTS_ROOT, base_dir=EXPERIMENT_ID))
print(f"Checkpoint round-trip verified: {RESULTS['checkpoint_path']}")
print(f"Saved artifacts: {EXPERIMENT_DIR}")
print(f"Created results archive: {RESULTS_ZIP}")


## 17. Download results

The ZIP is downloaded automatically during Run All. It contains the complete experiment folder. To download the selected trained checkpoint alone after Run All, execute the standalone helper below.

In [ ]:
def download_checkpoint() -> None:
    from google.colab import files as colab_files
    checkpoint_path = Path(RESULTS["checkpoint_path"])
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    colab_files.download(str(checkpoint_path))

if not RESULTS_ZIP.is_file():
    raise FileNotFoundError(f"Results ZIP not found: {RESULTS_ZIP}")
colab_files.download(str(RESULTS_ZIP))
print("To download the selected checkpoint separately, run: download_checkpoint()")


## 18. Scientific limitations and conclusion

- The reference dataset has 602 exact-row-deduplicated observations; unique rows do not confirm independent patients.
- No patient IDs, source-level linkage, or independent external validation are available here.
- The physiological mismatch and ordered coupling are model design hypotheses, not established causal or biological mechanisms.
- Calibration is descriptive and bin-dependent. No post-hoc calibrator or threshold optimization is fitted.
- Nested standard deviations describe variation across the configured random seeds on the same row-level dataset; they are not patient-clustered confidence intervals.
- This experiment does not establish clinical utility, safety, readiness, or superiority.
- PMRC-Net v1.0 and v1.1 remain distinct. This notebook runs v1.1 only and does not change or overwrite any historical project artifacts.

Do not interpret the synthetic software sanity checks as model performance. Interpret reported experiment metrics only after a complete run with the intended uploaded CSV and selected evaluation mode.